# Control and inspect Q_r minimization

This notebook calls the same shared function used by the KDE experiments. It reports the interval minimum, every detected interior local minimum, endpoint comparisons, and search diagnostics.

The central defaults live in [energy_minimization.py](../kde_tools/energy_minimization.py). Change `EnergySearchConfig` there to change defaults across experiments; an explicit configuration below overrides them for this notebook. The Gaussian objective and public entry point are in [qhat.py](../kde_tools/qhat.py).

Refinement uses [SciPy Brent root finding](https://docs.scipy.org/doc/scipy/reference/generated/scipy.optimize.brentq.html). The finite scan is a numerical search, not an exhaustive global-optimality certificate.


In [ ]:
from pathlib import Path
import hashlib
import json
import os
import sys

KDE_DIR = next(candidate.resolve()
    for ancestor in (Path.cwd(), *Path.cwd().parents)
    for candidate in (ancestor, ancestor / "Code" / "Code_kde")
    if (candidate / "kde_tools/pdf_export.py").is_file())
if str(KDE_DIR) not in sys.path:
    sys.path.insert(0, str(KDE_DIR))
os.environ.setdefault("MPLCONFIGDIR", str(KDE_DIR / ".matplotlib-cache"))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from kde_tools import EnergySearchConfig, minimize_qhat, energy_search_settings
from kde_tools.qhat import GaussianQhatObjective
from kde_tools.pdf_export import enable_automatic_pdf_saving

NOTEBOOK_NAME = "qhat_minimization_control"
pdf_directory = enable_automatic_pdf_saving(NOTEBOOK_NAME)


In [ ]:
# These are the only numerical-search controls needed in this notebook.
SEARCH = EnergySearchConfig()
# Example override: SEARCH = EnergySearchConfig(scan_points=513, scan_backend="exact")
H_MIN, H_MAX = 0.01, 2.0
R_VALUES = (0.0, 13/14, 1.0)

from kde_tools.experiment_inputs import REFERENCE_SAMPLE_PATH, load_reference_sample
sample_path = REFERENCE_SAMPLE_PATH
samples, _ = load_reference_sample()
objective = GaussianQhatObjective(samples)  # Share pair distances and FFT across r values.
analyses = {str(r): minimize_qhat(objective, H_MIN, H_MAX, r=r, config=SEARCH)
            for r in R_VALUES}

summary = pd.DataFrame([dict(
    r=r, local_minima=len(analyses[str(r)]["local_minima"]),
    interval_minimum_h=analyses[str(r)]["global_minimum"]["h"],
    interval_minimum_energy=analyses[str(r)]["global_minimum"]["energy"],
    boundary=analyses[str(r)]["global_minimum"]["boundary"],
    resolution_stable=analyses[str(r)]["diagnostics"]["resolution_stable"])
    for r in R_VALUES])
display(summary)


In [ ]:
# Every local minimum includes `is_global_minimum`, using the configured
# comparison tolerances and all detected candidates plus both endpoints.
local_minima = pd.DataFrame([dict(r=r, **point) for r in R_VALUES
    for point in analyses[str(r)]["local_minima"]])
display(local_minima)
print("Search settings:", energy_search_settings(SEARCH))

settings = dict(search=energy_search_settings(SEARCH), h_min=H_MIN, h_max=H_MAX,
                r_values=R_VALUES, samples_sha256=hashlib.sha256(samples.tobytes()).hexdigest())
run_id = hashlib.sha256(json.dumps(settings, sort_keys=True).encode()).hexdigest()[:12]
output_directory = KDE_DIR / "outputs/data" / NOTEBOOK_NAME / run_id
output_directory.mkdir(parents=True, exist_ok=True)
(output_directory / "analysis.json").write_text(json.dumps(
    dict(settings=settings, analyses=analyses), indent=2)+"\n")
summary.to_csv(output_directory / "summary.csv", index=False)
local_minima.to_csv(output_directory / "local_minima.csv", index=False)


In [ ]:
h = np.geomspace(H_MIN, H_MAX, 601)
fig, axes = plt.subplots(1, len(R_VALUES), figsize=(13, 4.2), constrained_layout=True)
for ax, r in zip(np.atleast_1d(axes), R_VALUES):
    report = analyses[str(r)]
    ax.loglog(h, [objective.log_energy(np.log(x), r)[0] for x in h], color="#24658a")
    for i, point in enumerate(report["local_minima"]):
        ax.scatter(point["h"], point["energy"], color="#d88321", s=40,
                   label="Interior local minima" if i == 0 else None, zorder=3)
    best = report["global_minimum"]
    ax.scatter(best["h"], best["energy"], color="#744da9", marker="*", s=115,
               label="Interval minimum", zorder=4)
    ax.set(xlabel="Bandwidth h", ylabel="Energy", title=f"r = {r:.6g}")
    ax.legend(fontsize=8)
fig.savefig(output_directory / "energy_minima.png", dpi=160)
plt.show()  # The shared exporter also saves this figure as a PDF.
plt.close(fig)
print("Full reports:", output_directory / "analysis.json")
print("Automatic PDF exports:", pdf_directory)
